In [7]:
from zigzag import zigzag, inverse_zigzag
import cv2
import numpy as np

In [8]:
np.set_printoptions(suppress=True)

In [9]:
# Quantization matrix (Q50)
base_q_mat = np.asarray([[16, 11, 10, 16,  24, 40,   51,  61],
                    [12, 12, 14, 19,  26, 58,   60,  55],
                    [14, 13, 16, 24,  40, 57,   69,  56],
                    [14, 17, 22, 29,  51, 87,   80,  62],
                    [18, 22, 37, 56,  68, 109, 103,  77],
                    [24, 36, 55, 64,  81, 104, 113,  92],
                    [49, 64, 78, 87, 103, 121, 120, 101],
                    [72, 92, 95, 98, 112, 100, 103,  99]
                  ], dtype = np.float64)

In [10]:
def custom_q_mat(Q):
    if Q < 50:
        S = 5000 / Q
    else:
        S = 200 - 2 * Q
    Ts = np.floor((base_q_mat * S + 50) / 100)
    Ts[Ts < 1] = 1
    return Ts.astype(np.uint8)

In [11]:
# Transform spatial to frequency domain
def transform_to_freq(imgarr, q_mat):
    sorted_coefficients = []
    h, w = imgarr.shape
    block_size = 8
    
    for i in range(0, h, block_size):
        for j in range(0, w, block_size):
            # Partition image into 8 x 8 blocks
            block = imgarr[i:i+block_size, j:j+block_size]
            # Shift values by 128
            block_f = np.float64(block) - 128.0  
            # DCT transform
            coeff = cv2.dct(block_f) 
            # Quantization
            q_coeff = np.around((coeff / q_mat).astype(np.float64))
            # Zigzag scan
            zigzag_scan = zigzag(q_coeff)
            sorted_coefficients.append(zigzag_scan)
            print(f"Block ({i//8}, {j//8}) DCT Coefficients:\n{q_coeff}\n")

    return sorted_coefficients

In [12]:
def reconstruct_image(sorted_coefficients, q_mat, img_shape):
    h, w = img_shape
    block_size = 8
    stego_img = np.zeros((h, w))
    idx = 0
    for i in range(0, h, block_size):
        for j in range(0, w, block_size):
            # Inverse zigzag → matriks 8×8
            zz = sorted_coefficients[idx]
            block_q = inverse_zigzag(zz, 8, 8)
            # Dequantization
            block_deq = block_q.astype(np.float64) * q_mat
            # Inverse DCT
            block_spatial = cv2.idct(block_deq)
            # Shift back by +128
            block_spatial += 128.0
            stego_img[i:i+8, j:j+8] = block_spatial
            idx += 1

    # Normalize and convert
    return np.uint8(np.clip(np.round(stego_img), 0, 255))